# Deep-dive: only_excel INN vs Озеро (SA/QP, null agr_id)

Цель тетрадки:
1. Взять `only_excel` INN (Excel minus merchants snapshot на 1-е число месяца).
2. Проверить, есть ли кейсы `only QP` (без SA).
3. Проверить, совпадают ли номера договоров из `agreements` у записей с `abs_agr_id is null`.
4. Посчитать, как меняется количество клиентов в срезе merchants, если убрать условие `agr_id is not null`.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = "/home/jovyan/documents/Equaring/Проверки/отчет_февраль_2026.xlsx"
excel_inn_col = "ИНН"
excel_contract_col = "Номер договора"
excel_name_col = "Наименование"

month_start = "2026-02-01"
merchants_snapshot_table = "sandbox_ai.stg__chesnov_aef__sa_acquiring_merchants"

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
def normalize_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    return s if re.fullmatch(r"\d+", s) else None

def normalize_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s

def to_sql_in_list(vals):
    safe = [str(v).replace("'", "''") for v in vals if pd.notna(v)]
    return ",".join([f"'{x}'" for x in safe])

## 1) Загружаем Excel и merchants snapshot, формируем only_excel INN

In [ ]:
df_excel = pd.read_excel(excel_path)
for col in [excel_inn_col, excel_contract_col, excel_name_col]:
    if col not in df_excel.columns:
        raise ValueError(f"В Excel не найдена колонка: {col}")

excel_norm = pd.DataFrame({
    'inn': df_excel[excel_inn_col].apply(normalize_id),
    'excel_contract_number': df_excel[excel_contract_col].apply(normalize_str),
    'excel_client_name': df_excel[excel_name_col].apply(normalize_str),
}).dropna(subset=['inn'])

with imp:
    dm_inn = imp.fetch(f"""
        select distinct cast(inn as string) as inn
        from {merchants_snapshot_table}
        where cast(d_valid_from as date) <= cast('{month_start}' as date)
          and (
                d_valid_to is null
                or cast(d_valid_to as date) >= cast('{month_start}' as date)
              )
          and inn is not null
    """)

set_excel_inn = set(excel_norm['inn'].dropna().tolist())
set_dm_inn = set(dm_inn['inn'].apply(normalize_id).dropna().tolist())

only_excel_inn = sorted(list(set_excel_inn - set_dm_inn))
only_excel_full = (
    excel_norm[excel_norm['inn'].isin(only_excel_inn)]
    [['inn', 'excel_client_name', 'excel_contract_number']]
    .drop_duplicates()
    .copy()
)

print('excel_unique_inn =', len(set_excel_inn))
print('merchants_unique_inn_on_month_start =', len(set_dm_inn))
print('only_excel_inn_cnt =', len(only_excel_inn))
display(only_excel_full.head(50))

## 1.5) Проверка only_excel INN в clients по ключу ИНН + Наименование

Проверяем:
1. Сколько INN из only_excel вообще находятся в `ods_alpha.scd1_companies`.
2. Совпадает ли ключ `ИНН + Наименование` между Excel и clients.
3. Выводим 10 примеров, где для одинакового ИНН название не сходится.

In [ ]:
# 1) Сколько INN из only_excel есть в clients
if len(only_excel_inn) == 0:
    clients_dim = pd.DataFrame(columns=['inn','n_cmp','c_cmp_name'])
else:
    in_inn = to_sql_in_list(only_excel_inn)
    with imp:
        clients_dim = imp.fetch(f"""
            select
                cast(c_inn as string) as inn,
                cast(n_cmp as string) as n_cmp,
                cast(c_cmp_name as string) as c_cmp_name
            from ods_alpha.scd1_companies
            where cast(c_inn as string) in ({in_inn})
        """)

clients_dim['inn'] = clients_dim['inn'].apply(normalize_id)
clients_dim['c_cmp_name_norm'] = clients_dim['c_cmp_name'].apply(normalize_str)
clients_dim = clients_dim.dropna(subset=['inn']).drop_duplicates()

clients_inn_set = set(clients_dim['inn'].dropna().tolist())
inn_presence_summary = pd.DataFrame([{
    'only_excel_inn_total': len(only_excel_inn),
    'only_excel_inn_found_in_clients': len(set(only_excel_inn) & clients_inn_set),
    'only_excel_inn_not_found_in_clients': len(set(only_excel_inn) - clients_inn_set),
}])

display(inn_presence_summary)

# 2) Сверка ключа INN + Наименование
excel_name_key = (
    only_excel_full[['inn', 'excel_client_name']]
    .dropna(subset=['inn'])
    .copy()
)
excel_name_key['excel_client_name_norm'] = excel_name_key['excel_client_name'].apply(normalize_str)
excel_name_key = excel_name_key.drop_duplicates()

clients_name_agg = (
    clients_dim.groupby('inn', as_index=False)
    .agg(c_cmp_name_candidates=('c_cmp_name_norm', lambda x: sorted({v for v in x if pd.notna(v)})))
)

name_check = excel_name_key.merge(clients_name_agg, on='inn', how='left')

# если INN не найден в clients -> candidates NaN
name_check['inn_found_in_clients'] = name_check['c_cmp_name_candidates'].notna()

# совпадение названия: excel_name входит в список candidates
name_check['name_match_for_same_inn'] = name_check.apply(
    lambda r: (r['excel_client_name_norm'] in r['c_cmp_name_candidates']) if isinstance(r['c_cmp_name_candidates'], list) else False,
    axis=1
)

# только случаи, где INN найден, но название не совпало
name_mismatch = name_check[
    name_check['inn_found_in_clients'] & (~name_check['name_match_for_same_inn'])
].copy()

name_mismatch['clients_name_candidates_str'] = name_mismatch['c_cmp_name_candidates'].apply(
    lambda x: '; '.join(x) if isinstance(x, list) else None
)

name_match_summary = pd.DataFrame([{
    'rows_checked_excel_inn_name': len(name_check),
    'rows_inn_found_in_clients': int(name_check['inn_found_in_clients'].sum()),
    'rows_inn_name_match': int(name_check['name_match_for_same_inn'].sum()),
    'rows_inn_name_mismatch': len(name_mismatch),
}])

display(name_match_summary)

# 3) 10 примеров mismatch по одинаковому INN
name_mismatch_top10 = name_mismatch[
    ['inn', 'excel_client_name', 'clients_name_candidates_str']
].head(10)

print('Top-10 mismatches for same INN (Excel name vs clients c_cmp_name):')
display(name_mismatch_top10)

## 2) Проверка: есть ли примеры только QP (без SA)

In [ ]:
if len(only_excel_inn) == 0:
    class_mix = pd.DataFrame(columns=['inn','agr_cnt','sa_cnt','qp_cnt','only_qp_flag'])
else:
    in_inn = to_sql_in_list(only_excel_inn)
    with imp:
        class_mix = imp.fetch(f"""
            with base as (
                select
                    cast(c.c_inn as string) as inn,
                    cast(a.acq_class as string) as acq_class,
                    cast(a.c_agr_number as string) as c_agr_number
                from ods_alpha.scd1_companies c
                join ods_alpha.scd1_agreements a
                  on a.n_cmp_client = c.n_cmp
                where cast(c.c_inn as string) in ({in_inn})
            ),
            agg as (
                select
                    inn,
                    count(*) as agr_cnt,
                    sum(case when acq_class = 'SA' then 1 else 0 end) as sa_cnt,
                    sum(case when acq_class = 'QP' then 1 else 0 end) as qp_cnt
                from base
                group by inn
            )
            select
                inn,
                agr_cnt,
                sa_cnt,
                qp_cnt,
                case when sa_cnt = 0 and qp_cnt > 0 then 1 else 0 end as only_qp_flag
            from agg
            order by only_qp_flag desc, qp_cnt desc, agr_cnt desc
        """)

display(class_mix.head(200))
only_qp = class_mix[class_mix['only_qp_flag'] == 1]
print('INN with only QP (no SA) =', len(only_qp))
display(only_qp.head(100))

## 3) Проверка: совпадают ли номера договоров у записей с `abs_agr_id is null`

In [ ]:
pairs = only_excel_full.dropna(subset=['inn', 'excel_contract_number']).copy()
pairs['excel_contract_number'] = pairs['excel_contract_number'].astype(str).str.strip()

if len(only_excel_inn) == 0:
    agr_null_id = pd.DataFrame(columns=['inn','c_agr_number','agr_id','acq_class','d_valid_from','d_valid_to'])
else:
    in_inn = to_sql_in_list(only_excel_inn)
    with imp:
        agr_null_id = imp.fetch(f"""
            select
                cast(c.c_inn as string) as inn,
                cast(a.c_agr_number as string) as c_agr_number,
                cast(a.abs_agr_id as string) as agr_id,
                cast(a.acq_class as string) as acq_class,
                cast(a.d_valid_from as date) as d_valid_from,
                cast(a.d_valid_to as date) as d_valid_to
            from ods_alpha.scd1_companies c
            join ods_alpha.scd1_agreements a
              on a.n_cmp_client = c.n_cmp
            where cast(c.c_inn as string) in ({in_inn})
              and a.abs_agr_id is null
        """)

agr_null_id['c_agr_number'] = agr_null_id['c_agr_number'].astype(str).str.strip()
pairs_cmp = pairs.rename(columns={'excel_contract_number': 'c_agr_number'})

check_null_match = pairs_cmp.merge(
    agr_null_id[['inn','c_agr_number','acq_class','d_valid_from','d_valid_to']],
    on=['inn','c_agr_number'],
    how='left'
)
check_null_match['matched_in_agreements_with_null_agr_id'] = check_null_match['acq_class'].notna()

summary_null = pd.DataFrame([{
    'pairs_in_excel': len(pairs_cmp),
    'matched_pairs_with_null_agr_id': int(check_null_match['matched_in_agreements_with_null_agr_id'].sum()),
    'not_matched_pairs': int((~check_null_match['matched_in_agreements_with_null_agr_id']).sum())
}])

display(summary_null)
display(check_null_match.head(200))

## 4) Сколько клиентов будет в merchants-срезе, если убрать `agr_id is not null`

In [ ]:
with imp:
    dm_cnt_compare = imp.fetch(f"""
        with base as (
            select
                cast(inn as string) as inn,
                cast(agr_id as string) as agr_id
            from {merchants_snapshot_table}
            where cast(d_valid_from as date) <= cast('{month_start}' as date)
              and (
                    d_valid_to is null
                    or cast(d_valid_to as date) >= cast('{month_start}' as date)
                  )
        )
        select
            count(distinct inn) as inn_without_agr_filter,
            count(distinct case when agr_id is not null then inn end) as inn_with_agr_not_null,
            count(distinct case when agr_id is null then inn end) as inn_with_agr_null_only
        from base
    """)

display(dm_cnt_compare)

## 5) (Опционально) SA/QP профиль для INN с `agr_id is null`

In [ ]:
with imp:
    null_agr_inn_profile = imp.fetch(f"""
        with null_inn as (
            select distinct cast(inn as string) as inn
            from {merchants_snapshot_table}
            where cast(d_valid_from as date) <= cast('{month_start}' as date)
              and (
                    d_valid_to is null
                    or cast(d_valid_to as date) >= cast('{month_start}' as date)
                  )
              and agr_id is null
              and inn is not null
        )
        select
            n.inn,
            sum(case when a.acq_class = 'SA' then 1 else 0 end) as sa_cnt,
            sum(case when a.acq_class = 'QP' then 1 else 0 end) as qp_cnt,
            count(*) as total_agr_cnt
        from null_inn n
        join ods_alpha.scd1_companies c
          on cast(c.c_inn as string) = n.inn
        join ods_alpha.scd1_agreements a
          on a.n_cmp_client = c.n_cmp
        group by n.inn
        order by total_agr_cnt desc
        limit 200
    """)

display(null_agr_inn_profile)

## 6) Проверка уникальности ИНН в обоих источниках

Проверяем, что в каждом источнике (`Excel` и `merchants snapshot`) один ИНН не повторяется в рамках набора уникальных ключей.
Если повторяется — выводим топ-100 ИНН с количеством повторов.

In [ ]:
excel_inn_dup = (
    excel_norm.groupby('inn', as_index=False)
    .size()
    .rename(columns={'size': 'cnt'})
    .sort_values('cnt', ascending=False)
)
excel_inn_dup = excel_inn_dup[excel_inn_dup['cnt'] > 1]

# merchants snapshot (на 1-е число) с учетом тех же условий, что выше
with imp:
    dm_inn_raw = imp.fetch(f"""
        select cast(inn as string) as inn
        from {merchants_snapshot_table}
        where cast(d_valid_from as date) <= cast('{month_start}' as date)
          and (
                d_valid_to is null
                or cast(d_valid_to as date) >= cast('{month_start}' as date)
              )
          and inn is not null
    """)

dm_inn_raw['inn'] = dm_inn_raw['inn'].apply(normalize_id)
dm_inn_dup = (
    dm_inn_raw.dropna(subset=['inn'])
    .groupby('inn', as_index=False)
    .size()
    .rename(columns={'size': 'cnt'})
    .sort_values('cnt', ascending=False)
)
dm_inn_dup = dm_inn_dup[dm_inn_dup['cnt'] > 1]

uniq_summary = pd.DataFrame([
    {
        'source': 'excel',
        'unique_inn_cnt': excel_norm['inn'].dropna().nunique(),
        'duplicated_inn_cnt': len(excel_inn_dup),
        'is_unique_inn_only': len(excel_inn_dup) == 0
    },
    {
        'source': 'merchants_snapshot_active_on_1st',
        'unique_inn_cnt': dm_inn_raw['inn'].dropna().nunique(),
        'duplicated_inn_cnt': len(dm_inn_dup),
        'is_unique_inn_only': len(dm_inn_dup) == 0
    }
])

display(uniq_summary)

print('Excel duplicated INN (top 100):')
display(excel_inn_dup.head(100))

print('Merchants duplicated INN (top 100):')
display(dm_inn_dup.head(100))

## 7) Совпадение `agr_id` для ИНН, которые есть в обоих источниках

Считаем:
- сколько общих ИНН между Excel и merchants;
- для скольких общих ИНН есть пересечение по `agr_id`;
- для скольких общих ИНН `agr_id` полностью расходятся.

In [ ]:
# excel inn->agr_id множества
excel_inn_agr = (
    excel_norm[['inn']].copy()
)
if 'agr_id' not in excel_norm.columns:
    # если в текущем df нет agr_id, пробуем взять из исходного Excel через ID договора
    if 'ID договора' in df_excel.columns:
        excel_inn_agr['agr_id'] = df_excel['ID договора'].apply(normalize_id)
        excel_inn_agr['inn'] = df_excel[excel_inn_col].apply(normalize_id)
    else:
        excel_inn_agr['agr_id'] = None
else:
    excel_inn_agr['agr_id'] = excel_norm['agr_id']

excel_inn_agr = excel_inn_agr.dropna(subset=['inn', 'agr_id']).drop_duplicates()

# merchants inn->agr_id множества на 1-е число
with imp:
    dm_inn_agr_raw = imp.fetch(f"""
        select
            cast(inn as string) as inn,
            cast(agr_id as string) as agr_id
        from {merchants_snapshot_table}
        where cast(d_valid_from as date) <= cast('{month_start}' as date)
          and (
                d_valid_to is null
                or cast(d_valid_to as date) >= cast('{month_start}' as date)
              )
          and inn is not null
          and agr_id is not null
    """)

dm_inn_agr_raw['inn'] = dm_inn_agr_raw['inn'].apply(normalize_id)
dm_inn_agr_raw['agr_id'] = dm_inn_agr_raw['agr_id'].apply(normalize_id)
dm_inn_agr = dm_inn_agr_raw.dropna(subset=['inn', 'agr_id']).drop_duplicates()

common_inn = sorted(list(set(excel_inn_agr['inn']) & set(dm_inn_agr['inn'])))

rows = []
for inn in common_inn:
    excel_set = set(excel_inn_agr.loc[excel_inn_agr['inn'] == inn, 'agr_id'].tolist())
    dm_set = set(dm_inn_agr.loc[dm_inn_agr['inn'] == inn, 'agr_id'].tolist())
    intersect = excel_set & dm_set

    rows.append({
        'inn': inn,
        'excel_agr_cnt': len(excel_set),
        'merchants_agr_cnt': len(dm_set),
        'intersect_agr_cnt': len(intersect),
        'has_any_agr_match': len(intersect) > 0
    })

agr_match_by_inn = pd.DataFrame(rows)

if len(agr_match_by_inn) == 0:
    agr_match_summary = pd.DataFrame([{
        'common_inn_cnt': 0,
        'inn_with_any_agr_match': 0,
        'inn_with_no_agr_match': 0,
        'inn_with_any_agr_match_pct': 0.0
    }])
else:
    agr_match_summary = pd.DataFrame([{
        'common_inn_cnt': len(agr_match_by_inn),
        'inn_with_any_agr_match': int(agr_match_by_inn['has_any_agr_match'].sum()),
        'inn_with_no_agr_match': int((~agr_match_by_inn['has_any_agr_match']).sum()),
        'inn_with_any_agr_match_pct': round(100.0 * agr_match_by_inn['has_any_agr_match'].mean(), 2)
    }])

display(agr_match_summary)

print('INN without agr_id matches (top 100):')
display(agr_match_by_inn[~agr_match_by_inn['has_any_agr_match']].head(100))

print('INN with agr_id matches (top 100):')
display(agr_match_by_inn[agr_match_by_inn['has_any_agr_match']].head(100))